# 实验 G：局部几何配对与生态位条件对照

留出 1、2、3，每折独立训练 real / zero / shuffle，共 9 次训练。每组保持 V1 的模型和 100 epochs，配对在三组间共享，并逐细胞重放 S 的来源与目标深度。先同步已完成 S 的 generated.h5ad 和 run_config.json。

使用新内核运行。无可信同类局部候选时不生成训练配对；推理保留来源 xy、表达和类型，输出 source_anchor 标记，仍纳入评价。shuffle 在同类、同可见切片内固定置换。

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from pathlib import Path
import sys

# 支持从仓库根目录、HM_DeepSpital 或 examples 启动。
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deepspatial/core.py").is_file() or (p / "HM_DeepSpital/deepspatial/core.py").is_file())
if (PROJECT / "HM_DeepSpital").is_dir():
    PROJECT = PROJECT / "HM_DeepSpital"
sys.path.insert(0, str(PROJECT))
from examples.run_g import run_experiment


In [ ]:
SOURCE_ROOT = PROJECT / "outputs/s_holdouts_1_2_3"
OUTPUT_ROOT = PROJECT / "outputs/g_holdouts_1_2_3"
CONFIG = PROJECT / "configs/merfish_g.json"
HOLDOUTS = [1, 2, 3]
MODES = ["real", "zero", "shuffle"]
DEVICE = "auto"
PAIRING_ONLY = False  # True 只检查配对；正式运行请使用另一全新输出目录。


In [ ]:
completed = run_experiment(
    source_root=SOURCE_ROOT, output_root=OUTPUT_ROOT, config_path=CONFIG,
    holdouts=HOLDOUTS, modes=MODES, device=DEVICE, pairing_only=PAIRING_ONLY,
)
completed


输出：每折 shared/pair_indices.npz、matching_support.csv、matching_by_type.csv，以及 real/zero/shuffle 各自的模型、日志和 generated.h5ad。配置中保存配对、初始权重、S 调度及实现哈希。

初始物理参数是预声明的候选值，未按隐藏切片调参：100 µm 层间距采用 300 µm 半径 / 150 µm 空间成本尺度；50 µm 间距同比缩半。作用于全部类型。稀有类评价继续使用可见训练丰度 ≤5%。质量结论需待完整运行和逐类评价后给出。